In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install -q transformers scikit-learn pandas

In [2]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
CSV_PATH = "/kaggle/input/datasets/jessypinkman47/mimic-iii-mean/llm_Train.csv"
VAL_PATH="/kaggle/input/datasets/jessypinkman47/mimic-iii-mean/llm_Val.csv"
TEST_PATH="/kaggle/input/datasets/jessypinkman47/mimic-iii-mean/llm_Test.csv"

MAX_LENGTH = 512   # BERT max is 512
BATCH_SIZE = 16
EPOCHS = 5
LR = 2e-5

In [4]:
df = pd.read_csv(CSV_PATH)

# def modify_clinical_text(text):
#     return f"""ICU admission summary.
# First 48 hours.
# 6-hour averaged measurements:

# {text}
# """

# df["clinical_text"] = df["clinical_text"].apply(modify_clinical_text)

df.head()

,patient_name,mortality_label,clinical_text
0,12797_episode1_timeseries.csv,0,Diastolic blood pressure 65.000. Glascow coma ...
1,9027_episode1_timeseries.csv,0,Diastolic blood pressure 56.185. Fraction insp...
2,40386_episode1_timeseries.csv,0,Diastolic blood pressure 46.052. Fraction insp...
3,48770_episode1_timeseries.csv,0,Diastolic blood pressure 81.574. Glascow coma ...
4,14037_episode1_timeseries.csv,0,Diastolic blood pressure 66.067. Glascow coma ...


In [9]:
# def modify_clinical_text(text):
#     lines = text.split("\n")
    
#     filtered_lines = []
    
#     for line in lines:
#         lower_line = line.lower()
        
#         if lower_line.startswith("weight:"):
#             continue
#         if lower_line.startswith("height:"):
#             continue
#         if lower_line.startswith("ph:"):
#             continue
        
#         filtered_lines.append(line)
    
#     filtered_text = "\n".join(filtered_lines)
    
#     return filtered_text

In [11]:
# df["clinical_text"] = df["clinical_text"].apply(modify_clinical_text)
# df.head()

,patient_name,mortality_label,clinical_text
0,12797_episode1_timeseries.csv,0,ICU admission summary.\nFirst 48 hours.\n6-hou...
1,9027_episode1_timeseries.csv,0,ICU admission summary.\nFirst 48 hours.\n6-hou...
2,40386_episode1_timeseries.csv,0,ICU admission summary.\nFirst 48 hours.\n6-hou...
3,48770_episode1_timeseries.csv,0,ICU admission summary.\nFirst 48 hours.\n6-hou...
4,14037_episode1_timeseries.csv,0,ICU admission summary.\nFirst 48 hours.\n6-hou...


In [5]:
# train_df, temp_df = train_test_split(
#     df,
#     test_size=0.3,
#     stratify=df["mortality_label"],
#     random_state=42
# )
train_df=df
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 14681
Val: 3222
Test: 3236


In [6]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print("GPU cleared")

GPU cleared


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

bert = AutoModel.from_pretrained(MODEL_NAME).to(device)

print("Model loaded.")

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Model loaded.


In [8]:
class ClinicalBertHybrid(nn.Module):
    def __init__(self, bert_model):
        super().__init__()
        self.bert = bert_model
        hidden_size = bert_model.config.hidden_size  # 768
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def mean_pooling(self, hidden, mask):
        mask = mask.unsqueeze(-1)
        return (hidden * mask).sum(1) / mask.sum(1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        #embedding
        
        pooled = self.mean_pooling(outputs.last_hidden_state, attention_mask)
        logits = self.classifier(pooled)
        
        return logits

In [9]:
class MortalityDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, "clinical_text"]
        label = self.df.loc[idx, "mortality_label"]

        encoding = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label": torch.tensor(label, dtype=torch.float)
        }

In [10]:
lengths = []

for text in df["clinical_text"]:
    tokens = tokenizer(text, truncation=False)["input_ids"]
    lengths.append(len(tokens))

print("Max length:", max(lengths))
print("Mean length:", sum(lengths)/len(lengths))
print("95th percentile:", sorted(lengths)[int(0.95*len(lengths))])

Max length: 172
Mean length: 146.08528029425787
95th percentile: 156


In [11]:
train_loader = DataLoader(MortalityDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(MortalityDataset(val_df), batch_size=BATCH_SIZE)
test_loader = DataLoader(MortalityDataset(test_df), batch_size=BATCH_SIZE)

In [12]:
model = ClinicalBertHybrid(bert).to(device)

# Freeze entire BERT
for param in model.bert.parameters():
    param.requires_grad = False

# for param in model.bert.encoder.layer[-4:].parameters():
#     param.requires_grad = True

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

In [13]:
def evaluate(loader):
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits).squeeze()

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    auc = roc_auc_score(all_labels, all_probs)
    acc = accuracy_score(all_labels, np.array(all_probs) > 0.5)

    return auc, acc

In [20]:
CHECKPOINT_PATH = "/kaggle/working/model_checkpoint.pt"
BEST_MODEL_PATH = "/kaggle/working/best_model.pt"
import os

In [21]:
start_epoch = 0
best_val_auc = 0

if os.path.exists(CHECKPOINT_PATH):
    print("Loading checkpoint...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_val_auc = checkpoint["best_val_auc"]

    print(f"Resuming from epoch {start_epoch}")

In [22]:
for epoch in range(start_epoch,EPOCHS):
    model.train()
    total_loss = 0

    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits.squeeze(), labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if (step + 1) % 50 == 0 or (step + 1) == len(train_loader):
            current_lr = optimizer.param_groups[0]['lr']
            avg_loss = total_loss / (step + 1)
            print(f"Step [{step+1}/{len(train_loader)}] "
                  f"Loss: {loss.item():.4f} "
                  f"Avg Loss: {avg_loss:.4f}"
                  f"LR: {current_lr:.6f}")

    if (epoch+1) % 2 == 0:
        val_auc, val_acc = evaluate(val_loader)
        print("Validation AUC:", val_auc)
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print("Best model saved.")
    # Save checkpoint
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_auc": best_val_auc
    }, CHECKPOINT_PATH)
    
    print(f"Checkpoint saved at epoch {epoch+1}")


    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

Step [50/918] Loss: 0.3791 Avg Loss: 0.4000LR: 0.000020
Step [100/918] Loss: 0.2517 Avg Loss: 0.4014LR: 0.000020
Step [150/918] Loss: 0.1147 Avg Loss: 0.3915LR: 0.000020
Step [200/918] Loss: 0.4438 Avg Loss: 0.3957LR: 0.000020
Step [250/918] Loss: 0.5029 Avg Loss: 0.3918LR: 0.000020
Step [300/918] Loss: 0.3479 Avg Loss: 0.3912LR: 0.000020
Step [350/918] Loss: 0.4942 Avg Loss: 0.3898LR: 0.000020
Step [400/918] Loss: 0.3231 Avg Loss: 0.3854LR: 0.000020
Step [450/918] Loss: 0.2067 Avg Loss: 0.3856LR: 0.000020
Step [500/918] Loss: 0.4165 Avg Loss: 0.3848LR: 0.000020
Step [550/918] Loss: 0.2737 Avg Loss: 0.3817LR: 0.000020
Step [600/918] Loss: 0.4720 Avg Loss: 0.3820LR: 0.000020
Step [650/918] Loss: 0.2737 Avg Loss: 0.3801LR: 0.000020
Step [700/918] Loss: 0.2046 Avg Loss: 0.3785LR: 0.000020
Step [750/918] Loss: 0.2722 Avg Loss: 0.3774LR: 0.000020
Step [800/918] Loss: 0.5686 Avg Loss: 0.3761LR: 0.000020
Step [850/918] Loss: 0.6757 Avg Loss: 0.3750LR: 0.000020
Step [900/918] Loss: 0.2596 Avg 

In [23]:

print("Loading best model for final test evaluation...")
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

test_auc, test_acc = evaluate(test_loader)

print("Final Test AUC:", test_auc)
print("Final Test Accuracy:", test_acc)


Loading best model for final test evaluation...
Final Test AUC: 0.8068420049552124
Final Test Accuracy: 0.8868974042027195
